# Chapter 8 · Generation and Inference

How to choose each character (greedy, temperature, top-k and top-p), how to measure the quality of what is generated, the key–value cache and memory. It uses the GPT trained in chapter 7, saved in `material/gpt_quixote.pt`; it trains nothing. PyTorch on the CPU. The whole notebook takes a couple of minutes.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Loading the model

The classes are those of chapters 6 and 7. In Colab, the cell downloads the model and the novel if they are not there.

In [1]:
"""Chapter 8: generation and inference with the Quixote GPT. CPU."""
import math
import re
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path(".")
MODEL = Path("material/gpt_quixote.pt")
if not MODEL.exists():  # in Colab, download it from the repository
    import urllib.request
    MODEL.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/gpt_quixote.pt", MODEL)
TEXT = Path("material/quixote.txt")
if not TEXT.exists():  # in Colab, download it from the repository
    import urllib.request
    TEXT.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quixote.txt", TEXT)
START = "In a village of La Mancha"


# --- The model of chapters 6 and 7 ---------------------------------------

class Head(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["context"]
        size = C // config["heads"]
        self.query = nn.Linear(C, size, bias=False)
        self.key = nn.Linear(C, size, bias=False)
        self.value = nn.Linear(C, size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        future = self.mask[:T, :T] == 0
        scores = scores.masked_fill(future, float("-inf"))
        return F.softmax(scores, dim=-1) @ v


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.heads = nn.ModuleList(
            [Head(config) for _ in range(config["heads"])])
        self.projection = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norm1, self.norm2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norm1(x)
        attention = torch.cat([head(h) for head in self.heads], dim=-1)
        x = x + self.projection(attention)
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.symbol_embedding = nn.Embedding(vocab_size, C)
        self.position_embedding = nn.Embedding(config["context"], C)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config["layers"])])
        self.norm = nn.LayerNorm(C)
        self.output = nn.Linear(C, vocab_size)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.symbol_embedding(ids) + self.position_embedding(positions)
        return self.output(self.norm(self.blocks(x)))


def load_model(path=MODEL):
    saved = torch.load(path)
    symbols = saved["symbols"]
    model = GPT(len(symbols), saved["config"])
    model.load_state_dict(saved["model"])
    model.eval()
    return model, symbols, {c: i for i, c in enumerate(symbols)}

In [2]:
torch.set_num_threads(4)
model, symbols, to_id = load_model()
print("Symbols:", len(symbols), "· parameters:", sum(p.numel() for p in model.parameters()))

Symbols: 90 · parameters: 215002


## 2. Temperature by hand

The logits of exercise 1, divided by three different temperatures.

In [3]:
logits = torch.tensor([3.0, 1.0, 0.0])
for temperature in (1.0, 0.5, 2.0):
    print(temperature, F.softmax(logits / temperature, dim=-1).round(decimals=3))

1.0 tensor([0.8440, 0.1140, 0.0420])
0.5 tensor([0.9800, 0.0180, 0.0020])
2.0 tensor([0.6290, 0.2310, 0.1400])


## 3. Choosing and generating

In [4]:
# --- How to choose the next character ---------------------------------------

def choose(logits, generator, temperature=1.0, k=None, p=None):
    """Choose an id from the logits of one position."""
    if temperature == 0:
        return int(torch.argmax(logits))  # greedy: always the most likely
    logits = logits / temperature
    if k is not None:  # top-k: only the k most likely
        threshold = torch.topk(logits, k).values[-1]
        logits = logits.masked_fill(logits < threshold, float("-inf"))
    probabilities = F.softmax(logits, dim=-1)
    if p is not None:  # top-p: the most likely until they add up to p
        order = torch.argsort(probabilities, descending=True)
        ordered = probabilities[order]
        before = torch.cumsum(ordered, dim=0) - ordered  # sum of those ahead
        probabilities[order[before >= p]] = 0
        probabilities = probabilities / probabilities.sum()
    return int(torch.multinomial(probabilities, 1, generator=generator))


@torch.no_grad()
def generate(model, symbols, to_id, start, n, seed=7, **strategy):
    generator = torch.Generator().manual_seed(seed)
    context = model.config["context"]
    ids = [to_id[c] for c in start]
    for _ in range(n):
        logits = model(torch.tensor([ids[-context:]]))[0, -1]
        ids.append(choose(logits, generator, **strategy))
    return "".join(symbols[i] for i in ids)

In [5]:
for strategy in (dict(temperature=0), dict(temperature=0.8), dict(temperature=1.5),
                 dict(temperature=1.0, k=5), dict(temperature=1.0, p=0.9)):
    print(strategy)
    print("   ", repr(generate(model, symbols, to_id, START, 150, **strategy)))

{'temperature': 0}
    'In a village of La Mancha come of the same of the same of the same\nof the same of the same of the same of the same of the same of the\nsame of the same of the same of the same '
{'temperature': 0.8}
    'In a village of La Mancha was already envy such a gallous look him if the in\nattraction a send brelief To Egerma did not able the master for oppen them, betney be\none. I never'
{'temperature': 1.5}
    'In a village of La Mancha?” wholice, serving mi stive thinqrockh\nJaisiticour atly abk aft sendhmen.”\n\n“Then arms did not,” replied,Y iw if SeckleiA tumN T0” I,’\n\n“Noo. I\nnot g'
{'temperature': 1.0, 'k': 5}
    'In a village of La Mancha want ore\nto breass in a give as trove that it, with the work as went there\nto be them in a country.”\n\n“Then this seak,” said Don Quixote, “it said to'
{'temperature': 1.0, 'p': 0.9}
    'In a village of La Mancha want breed served\nup to me. The look his brother wont.”\n\n“By my send better, I have have a comparial wh

## 4. Measuring quality

Fraction of real words and variety of pairs of words in 2000 characters per strategy. It takes about a minute and a half; the times depend on the computer.

In [6]:
# --- Measuring the quality of a generated text ------------------------------

def words(text):
    return re.findall(r"[a-zñ]+(?:’[a-zñ]+)?", text.lower())  # don’t, señor


def quality(text, quixote_words):
    word_list = words(text)
    real = sum(word in quixote_words for word in word_list) / len(word_list)
    pairs = list(zip(word_list, word_list[1:]))
    variety = len(set(pairs)) / len(pairs)
    return real, variety

In [7]:
quixote_words = set(words(TEXT.read_text(encoding="utf-8")))
strategies = {
    "greedy": dict(temperature=0),
    "temperature 0.5": dict(temperature=0.5),
    "temperature 0.8": dict(temperature=0.8),
    "temperature 1.0": dict(temperature=1.0),
    "temperature 1.5": dict(temperature=1.5),
    "top-k 5": dict(temperature=1.0, k=5),
    "top-p 0.9": dict(temperature=1.0, p=0.9),
}
table = {}
for name, strategy in strategies.items():
    begin = time.perf_counter()
    text = generate(model, symbols, to_id, START, 2000, **strategy)
    ms = 1000 * (time.perf_counter() - begin) / 2000
    real, variety = quality(text[len(START):], quixote_words)
    table[name] = (real, variety, ms)
    print("%-16s real words %.3f · variety %.3f · %.1f ms/char" % (
        name, real, variety, ms))
    print("   ", repr(text[:200]))

greedy           real words 1.000 · variety 0.008 · 5.9 ms/char
    'In a village of La Mancha come of the same of the same of the same\nof the same of the same of the same of the same of the same of the\nsame of the same of the same of the same of the same of the same\no'
temperature 0.5  real words 0.956 · variety 0.775 · 6.1 ms/char
    'In a village of La Mancha was already and found a good astorication\nhim, with the first and that the book of my carriers, and the master\nfor the interrant that with this never to be the stake on of th'
temperature 0.8  real words 0.880 · variety 0.948 · 6.5 ms/char
    'In a village of La Mancha was already envy such a gallous look him if the in\nattraction a send brelief To Egerma did not able the master for oppen them, betney be\none. I never to be did not on the per'
temperature 1.0  real words 0.794 · variety 0.979 · 6.1 ms/char
    'In a village of La Mancha want breed served\nup to me. The look his broth, in at read that send beverfed by gar

## 5. The key–value cache

In [8]:
# --- A key-value cache ------------------------------------------------------

@torch.no_grad()
def step_with_cache(model, symbol_id, position, cache):
    """Compute the logits of a new position, reusing the past."""
    x = (model.symbol_embedding.weight[symbol_id]
         + model.position_embedding.weight[position]).view(1, -1)
    for b, block in enumerate(model.blocks):
        h = block.norm1(x)
        outputs = []
        for c, head in enumerate(block.heads):
            keys, values = cache.setdefault((b, c), ([], []))
            keys.append(head.key(h))
            values.append(head.value(h))
            K, V = torch.cat(keys), torch.cat(values)
            q = head.query(h)
            weights = F.softmax(q @ K.T / math.sqrt(K.shape[-1]), dim=-1)
            outputs.append(weights @ V)
        x = x + block.projection(torch.cat(outputs, dim=-1))
        x = x + block.mlp(block.norm2(x))
    return model.output(model.norm(x))[0]


@torch.no_grad()
def generate_with_cache(model, symbols, to_id, start, n):
    """Greedy generation with a cache; the whole text fits in the context."""
    assert len(start) + n <= model.config["context"]
    cache, ids = {}, [to_id[c] for c in start]
    for position, symbol_id in enumerate(ids):
        logits = step_with_cache(model, symbol_id, position, cache)
    for _ in range(n):
        ids.append(int(torch.argmax(logits)))
        logits = step_with_cache(model, ids[-1], len(ids) - 1, cache)
    return "".join(symbols[i] for i in ids)


@torch.no_grad()
def check_cache(model, ids):
    """Largest difference between the logits with and without the cache."""
    full = model(torch.tensor([ids]))[0]
    cache = {}
    step_by_step = torch.stack(
        [step_with_cache(model, symbol_id, position, cache)
         for position, symbol_id in enumerate(ids)])
    return float((full - step_by_step).abs().max())


def time_it(function, repeats=5):
    begin = time.perf_counter()
    for _ in range(repeats):
        result = function()
    return result, (time.perf_counter() - begin) / repeats

In [9]:
ids = [to_id[c] for c in START + ", the name of which I have no desire"]
print("Max difference with and without cache:", check_cache(model, ids))
no_cache = lambda: generate(model, symbols, to_id, "I", 63, temperature=0)
with_cache = lambda: generate_with_cache(model, symbols, to_id, "I", 63)
text_without, time_without = time_it(no_cache)
text_with, time_with = time_it(with_cache)
print("Same text with and without cache:", text_without == text_with)
print("Without cache: %.1f ms · with cache: %.1f ms" % (1000 * time_without,
                                                        1000 * time_with))

Max difference with and without cache: 6.198883056640625e-06
Same text with and without cache: True
Without cache: 390.4 ms · with cache: 228.1 ms


## 6. Memory

In [10]:
# --- Memory ---------------------------------------------------------------

def parameter_memory(n_parameters, bytes_per_number=4):
    return n_parameters * bytes_per_number


def cache_memory(blocks, dimension, context, bytes_per_number=4):
    # two tables (keys and values) per block, each of context x dimension
    return 2 * blocks * dimension * context * bytes_per_number

In [11]:
n = sum(parameter.numel() for parameter in model.parameters())
print("Parameters:", n, "· memory:", parameter_memory(n), "bytes")
print("Cache with 64 positions:", cache_memory(4, 64, 64), "bytes")

Parameters: 215002 · memory: 860008 bytes
Cache with 64 positions: 131072 bytes


## Exercises

1. Compute, by hand or with a calculator, the softmax of the logits [3, 1, 0] with temperatures 1, 0.5 and 2. What happens to the probability of the first output?

2. What temperature makes the logits [3, 1, 0] give an almost uniform distribution? And an almost greedy one? Reason it out without calculating.

3. With the logits [3, 1, 0] and T=1, apply top-k with k=2 and top-p with p=0.9. How many outputs does each one keep?

4. Follow by hand the top-p code of `choose` with the probabilities [0.1, 0.6, 0.3] and p=0.5: what are the values of `order`, `ordered` and `before`? Which characters become zero?

5. Explain why the greedy choice falls into loops with our model and why sampling avoids them.

6. According to the chapter's table, which strategy would you choose to generate a story? And to complete a word automatically while someone is typing? Justify your answer with the two measures.

7. Add two entries to the dictionary `strategies` of `run`, with temperatures 0.3 and 1.2, run the program and add their rows to the table. Do they follow the same trend?

8. Repeat the rows of temperature 1, temperature 0.8, top-k and top-p with the seeds 1 to 5 (the parameter `seed` of `generate`). How much do the numbers change? Is the order between strategies kept?

9. The real-words measure counts as correct a word that exists in Don Quixote, even if it is badly used. Give an example of a text with 100% real words and no meaning. What other measure would you propose?

10. Explain in your own words why the KV cache does not change the result. What role does the causal mask play?

11. Call `generate_with_cache` with the start `"I"` and n=64. Why does the program stop? What would happen if we removed the `assert`?

12. How many positions does the model process without a cache to generate 127 characters from one, if the context were 128? And with a cache?

13. Compute the memory of the cache of our model if the context were 1024 positions. And that of the parameters if we stored each number with 2 bytes?

14. Table of quality, time and memory. With the results of your run, fill in a table with the seven strategies, their two quality measures and their time per character, and add the memory of the parameters and of the cache. Write five lines with your conclusions.

## References

The sampling function of [nanoGPT](https://github.com/karpathy/nanoGPT), by Andrej Karpathy, uses temperature and top-k as here. Top-p was proposed in *The Curious Case of Neural Text Degeneration* (Holtzman and colleagues, 2020). Text of *Don Quixote*, by Miguel de Cervantes, in John Ormsby's English translation, in the public domain. The code and the measures are original.